## GRU training on pose keypoints (Colab)

This notebook trains a GRU binary classifier (fall vs normal) on the prepared dataset in `./dataset_ready/`.

### Expected files
- `dataset_ready/X_train.npy`, `dataset_ready/y_train.npy`
- `dataset_ready/X_val.npy`, `dataset_ready/y_val.npy`
- `dataset_ready/X_test.npy`, `dataset_ready/y_test.npy`

### Value conventions
- **0**: human detected, but a specific keypoint is missing / low confidence (we store `0,0` for that keypoint).
- **-1**: no human detected (or all keypoints are unusable) for that frame.

### Masking
- We treat timesteps where **all features are -1** as padded/invalid.
- We compute per-sample sequence lengths from those rows.
- We replace `-1` with `0` before the GRU and use `pack_padded_sequence` so the GRU ignores invalid timesteps.


In [ ]:
# ===== Setup (Colab) =====
# If you're on Colab, torch/numpy are usually preinstalled.
# Uncomment if you need to install/upgrade.
# !pip -q install --upgrade torch numpy matplotlib

import os
import math
import json
import random
from dataclasses import dataclass

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)


In [ ]:
# ===== Load prepared dataset =====
DATA_DIR = './dataset_ready'
mask_value = -1.0

X_train = np.load(os.path.join(DATA_DIR, 'X_train.npy'))
y_train = np.load(os.path.join(DATA_DIR, 'y_train.npy'))
X_val = np.load(os.path.join(DATA_DIR, 'X_val.npy'))
y_val = np.load(os.path.join(DATA_DIR, 'y_val.npy'))
X_test = np.load(os.path.join(DATA_DIR, 'X_test.npy'))
y_test = np.load(os.path.join(DATA_DIR, 'y_test.npy'))

print('X_train', X_train.shape, X_train.dtype)
print('y_train', y_train.shape, y_train.dtype, 'pos%=', float(y_train.mean()) if y_train.size else None)
print('X_val  ', X_val.shape, X_val.dtype)
print('y_val  ', y_val.shape, y_val.dtype, 'pos%=', float(y_val.mean()) if y_val.size else None)
print('X_test ', X_test.shape, X_test.dtype)
print('y_test ', y_test.shape, y_test.dtype, 'pos%=', float(y_test.mean()) if y_test.size else None)

assert X_train.ndim == 3, 'Expected X to be (N, T, F)'
input_size = X_train.shape[-1]
print('input_size inferred:', input_size)


In [ ]:
# ===== Dataset / Dataloader with masking + packing lengths =====
# If you hit CUDA OOM on Colab GPU, reduce this back to 32.
BATCH_SIZE = 64

class NpySequenceDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx]).float()        # (T, F)
        y = torch.tensor(self.y[idx]).float()            # scalar (0/1)
        return x, y


def collate_with_lengths(batch, mask_value: float = -1.0):
    xs, ys = zip(*batch)
    x = torch.stack(xs, dim=0)  # (B, T, F)
    y = torch.stack(ys, dim=0)  # (B,)

    # valid timestep = not all features are mask_value
    valid_row = ~(x == mask_value).all(dim=-1)  # (B, T)
    lengths = valid_row.sum(dim=1).to(torch.int64)  # (B,)
    lengths = torch.clamp(lengths, min=1)

    # Replace masked values with 0 so GRU sees neutral input at masked positions.
    x = x.masked_fill(x == mask_value, 0.0)
    return x, y, lengths

train_ds = NpySequenceDataset(X_train, y_train)
val_ds = NpySequenceDataset(X_val, y_val)
test_ds = NpySequenceDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
                          collate_fn=lambda b: collate_with_lengths(b, mask_value=mask_value))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
                        collate_fn=lambda b: collate_with_lengths(b, mask_value=mask_value))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
                         collate_fn=lambda b: collate_with_lengths(b, mask_value=mask_value))

xb, yb, lb = next(iter(train_loader))
print('batch x', xb.shape, 'y', yb.shape, 'lengths', lb.shape, 'min/max', int(lb.min()), int(lb.max()))


In [ ]:
# ===== Model =====

@dataclass
class GRUConfig:
    input_size: int = 34
    hidden_size: int = 64
    num_layers: int = 2
    output_size: int = 1
    dropout_prob: float = 0.3

cfg = GRUConfig(input_size=int(input_size))
print(cfg)

class GRUNet(nn.Module):
    def __init__(self, cfg: GRUConfig):
        super().__init__()
        self.gru = nn.GRU(
            input_size=cfg.input_size,
            hidden_size=cfg.hidden_size,
            num_layers=cfg.num_layers,
            batch_first=True,
            dropout=cfg.dropout_prob if cfg.num_layers > 1 else 0.0,
            bidirectional=False,
        )
        self.fc = nn.Linear(cfg.hidden_size, cfg.output_size)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F), lengths: (B,)
        # pack (requires CPU lengths)
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h_n = self.gru(packed)
        # h_n: (num_layers, B, hidden_size) for unidirectional GRU
        last = h_n[-1]  # (B, hidden)
        logits = self.fc(last)  # (B, 1)
        return logits.squeeze(1)

model = GRUNet(cfg).to(DEVICE)
print(model)


In [ ]:
# ===== Training config =====

NUM_EPOCHS = 300
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
PATIENCE = 15
GRAD_CLIP_NORM = 1.0  # recommended for RNN stability

# Class imbalance handling: pos_weight = (#negative / #positive)
pos = float((y_train == 1).sum())
neg = float((y_train == 0).sum())
pos_weight_value = (neg / max(1.0, pos))
print('pos/neg:', pos, neg, 'pos_weight:', pos_weight_value)

pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Optional but recommended: reduce LR when val loss plateaus.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6
)

def batch_metrics(logits: torch.Tensor, y: torch.Tensor):
    # y is float {0,1}
    probs = torch.sigmoid(logits)
    preds = (probs >= 0.5).to(torch.int64)
    yt = y.to(torch.int64)

    correct = (preds == yt).sum().item()
    total = yt.numel()

    tp = ((preds == 1) & (yt == 1)).sum().item()
    tn = ((preds == 0) & (yt == 0)).sum().item()
    fp = ((preds == 1) & (yt == 0)).sum().item()
    fn = ((preds == 0) & (yt == 1)).sum().item()

    precision = tp / (tp + fp + 1e-12)
    recall = tp / (tp + fn + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    acc = correct / max(1, total)

    return {
        'acc': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
    }


In [ ]:
# ===== Train / Validate with Early Stopping =====

def run_epoch(model, loader, train: bool):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_n = 0

    agg = {'acc': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
    counts = {'tp': 0, 'tn': 0, 'fp': 0, 'fn': 0}

    for x, y, lengths in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        lengths = lengths.to(DEVICE)

        with torch.set_grad_enabled(train):
            logits = model(x, lengths)
            loss = criterion(logits, y)

            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                optimizer.step()

        bs = y.shape[0]
        total_loss += loss.item() * bs
        total_n += bs

        m = batch_metrics(logits.detach(), y.detach())
        for k in ['tp', 'tn', 'fp', 'fn']:
            counts[k] += int(m[k])

    # derive metrics from confusion counts
    tp, tn, fp, fn = counts['tp'], counts['tn'], counts['fp'], counts['fn']
    precision = tp / (tp + fp + 1e-12)
    recall = tp / (tp + fn + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    acc = (tp + tn) / max(1, (tp + tn + fp + fn))

    avg_loss = total_loss / max(1, total_n)
    return avg_loss, {
        'acc': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
    }

best_state = None
best_val_loss = float('inf')
no_improve = 0

# Optional: track best validation F1 too (we still early-stop on val loss)
best_val_f1 = -1.0

history = {
    'train_loss': [],
    'val_loss': [],
    'train_acc': [],
    'val_acc': [],
}

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_m = run_epoch(model, train_loader, train=True)
    val_loss, val_m = run_epoch(model, val_loader, train=False)

    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_m['acc'])
    history['val_acc'].append(val_m['acc'])

    lr = optimizer.param_groups[0]['lr']

    print(
        f"Epoch {epoch:03d} | lr={lr:.2e} | "
        f"train loss={train_loss:.4f} acc={train_m['acc']:.3f} f1={train_m['f1']:.3f} | "
        f"val loss={val_loss:.4f} acc={val_m['acc']:.3f} f1={val_m['f1']:.3f}"
    )

    if val_loss < best_val_loss - 1e-6:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_val_f1 = float(val_m['f1'])
        no_improve = 0
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch} (best val loss={best_val_loss:.4f})")
        break

# Restore best weights
if best_state is not None:
    model.load_state_dict(best_state)
    print('Restored best model weights.')


In [ ]:
# ===== Plot curves =====

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history['train_loss'], label='train')
ax[0].plot(history['val_loss'], label='val')
ax[0].set_title('Loss')
ax[0].set_xlabel('epoch')
ax[0].legend()

ax[1].plot(history['train_acc'], label='train')
ax[1].plot(history['val_acc'], label='val')
ax[1].set_title('Accuracy')
ax[1].set_xlabel('epoch')
ax[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# ===== Test evaluation =====

test_loss, test_m = run_epoch(model, test_loader, train=False)
print(f"Test loss={test_loss:.4f} acc={test_m['acc']:.3f} precision={test_m['precision']:.3f} recall={test_m['recall']:.3f} f1={test_m['f1']:.3f}")
print('Confusion:', {k: test_m[k] for k in ['tp','tn','fp','fn']})


In [ ]:
# ===== Confusion matrix (plot) + metrics table =====

cm = np.array([[test_m['tn'], test_m['fp']],
               [test_m['fn'], test_m['tp']]], dtype=np.int64)

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_title('Confusion matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['0 (normal)', '1 (fall)'])
ax.set_yticklabels(['0 (normal)', '1 (fall)'])

for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, str(v), ha='center', va='center', color='black')

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

metrics_table = {
    'split': 'test',
    'loss': float(test_loss),
    'acc': float(test_m['acc']),
    'precision': float(test_m['precision']),
    'recall': float(test_m['recall']),
    'f1': float(test_m['f1']),
    'tp': int(test_m['tp']),
    'tn': int(test_m['tn']),
    'fp': int(test_m['fp']),
    'fn': int(test_m['fn']),
}

print(json.dumps(metrics_table, indent=2))


In [ ]:
# ===== Save checkpoint =====

ckpt = {
    'model_state_dict': model.state_dict(),
    'cfg': cfg.__dict__,
    'mask_value': mask_value,
    'best_val_loss': best_val_loss,
    'history': history,
    'test_metrics': metrics_table,
}

ckpt_path = 'gru_pose_masked.pt'
torch.save(ckpt, ckpt_path)
print('Saved:', ckpt_path)
